In [ ]:
from aimet_torch import quantsim
import pickle as pkl
model_path = "/workspace/quant_pipeline/QuantizedSSR/quantized_export/vad_detector_int8/vad_detector_int8.pth"
config_path = "/workspace/quant_pipeline/QuantizedSSR/config/fully_symmetric.json"
config = "/workspace/quant_pipeline/QuantizedSSR/ssr/projects/configs/SSR_e2e.py"

import os
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import onnxruntime as ort
import torch
import torch.nn as nn
from onnxruntime_extensions import (
    PyCustomOpDef,
    enable_py_op,
    get_library_path,
    onnx_op,
)

from aimet_common.defs import QuantScheme
from aimet_torch import quantsim
from aimet_torch.v2.nn import QuantizationMixin
from aimet_torch.v2.quantsim import QuantizationSimModel

from evaluation.eval_dataset import extract_data, build_eval_loader
from ssr.projects.mmdet3d_plugin.SSR.utils.builder import build_model

def move_to_device(obj: Any, device: torch.device, non_blocking: bool = True) -> Any:
    if torch.is_tensor(obj):
        return obj.to(device, non_blocking=non_blocking)
    if isinstance(obj, list):
        return [move_to_device(x, device, non_blocking) for x in obj]
    if isinstance(obj, tuple):
        return tuple(move_to_device(x, device, non_blocking) for x in obj)
    if isinstance(obj, dict):
        return {k: move_to_device(v, device, non_blocking) for k, v in obj.items()}
    return obj


def prepare_batch(batch: Dict[str, Any], device: torch.device) -> Dict[str, Any]:
    return move_to_device(extract_data(batch), device)

class AimetTraceWrapper(nn.Module):
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model = model
        self.runtime_batch: Optional[Dict[str, Any]] = None

    def set_batch(self, batch: Dict[str, Any]) -> None:
        self.runtime_batch = batch

    def forward(self, img: Optional[torch.Tensor] = None, **kwargs):
        if kwargs:
            return self.model(img=img, **kwargs)

        if self.runtime_batch is None:
            raise RuntimeError("runtime_batch must be set before tracing")
        if img is None:
            raise ValueError("img must not be None in trace mode")

        # img = ensure_trace_img_shape(img)
        batch = self.runtime_batch

        return self.model.forward_trace(
            img_metas=batch["img_metas"],
            img=[img],
            ego_fut_cmd=batch["ego_fut_cmd"],
            ego_his_trajs=batch["ego_his_trajs"],
            ego_lcf_feat=batch["ego_lcf_feat"],
        )

def rebuild_model_with_encodings(
    quant_weights: str,
    config: Any,
    device: torch.device = "cpu",
    quant_scheme: Any = "tf_enhanced",
    default_output_bw: int=8,
    default_param_bw: int=8,
    config_path: Optional[str]=None,
    enable_bn_fold: bool=False,
    skip_layer_names: Optional[List[str]] = None,
) -> Dict[str, Any]:
    print("[TORCH] Rebuilding FP32 model and applying encodings...")

    cfg, dataset, data_loader = build_eval_loader(config)
    model = build_model(cfg.model, test_cfg=cfg.get("test_cfg"))

    print(f"[TORCH] Loading exported AIMET weights from: {quant_weights}")
    ckpt = torch.load(quant_weights, map_location="cpu")
    state_dict = ckpt.model.state_dict()

    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print(f"[TORCH] Missing keys: {len(missing)}")
    print(f"[TORCH] Unexpected keys: {len(unexpected)}")
    if missing:
        print("[TORCH] First missing keys:", missing[:20])
    if unexpected:
        print("[TORCH] First unexpected keys:", unexpected[:20])

    model.CLASSES = getattr(dataset, "CLASSES", None)
    model.PALETTE = getattr(dataset, "PALETTE", None)
    model = model.to(device).eval()

    first_batch = prepare_batch(next(iter(data_loader)), device)

    wrapped_model = AimetTraceWrapper(model=model).to(device).eval()
    wrapped_model.set_batch(first_batch)

    real_img = ensure_trace_img_shape(extract_single_img(first_batch))
    dummy_input = torch.zeros_like(real_img)

    maybe_run_bn_fold(wrapped_model, dummy_input, enable_bn_fold)

    sim = create_quant_sim(
        model=wrapped_model,
        device=device,
        dummy_input=dummy_input,
        quant_scheme=quant_scheme,
        default_output_bw=default_output_bw,
        default_param_bw=default_param_bw,
        config_path=config_path,
        skip_layer_names=skip_layer_names,
    )

    # print("has load_encodings:", hasattr(sim, "load_encodings"))
    # sim.load_encodings(encoding_path, strict=True, partial=False)

    # log_uninitialized_quantizers(sim)
    sim.model.to(device).eval()

    return {
        "backend": "torch",
        "model": sim.model,
        "sim": sim,
        "session": None,
        "input_name": None,
        "output_names": None,
        "graph_optimization_level": None,
    }

model_obj = rebuild_model_with_encodings(
    quant_weights=model_path,
    config=config,
    config_path=config_path,
)

print(model_obj["model"])


[TORCH] Rebuilding FP32 model and applying encodings...


2026-04-13 09:25:57.999 | WARNING  | ssr.projects.mmdet3d_plugin.datasets.builder:build_dataloader:80 - Only can be used for obtain inference speed!


[TORCH] Loading exported AIMET weights from: /workspace/quant_pipeline/QuantizedSSR/quantized_export/vad_detector_int8/vad_detector_int8.pth
[TORCH] Missing keys: 0
[TORCH] Unexpected keys: 0


NameError: name 'ensure_trace_img_shape' is not defined